In [11]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import random
import matplotlib.pyplot as plt



def select_alpha_range(target_daily_tasks):
    """根据目标日均任务数，选择合适的退化率范围"""
    if target_daily_tasks <= 5:  # 稀疏
        return 0.004, 0.006
    elif 6 <= target_daily_tasks <= 12:  # 中等
        return 0.010, 0.013
    else:  # 13+ 密集
        return 0.020, 0.025


def generate_maintenance_plan(N, T, random_seed, target_daily_tasks):
    """生成维护计划（新版本：基于目标任务密度）"""
    random.seed(random_seed)
    
    caissons = [i for i in range(N)]
    initial_p_values = [round(random.uniform(3.2, 3.5), 2) for _ in caissons]
    
    a, b = select_alpha_range(target_daily_tasks)
    alpha_values = [round(random.uniform(a, b), 6) for _ in caissons]
    
    columns = ['INDEX', 'CSEM', 'INITIALP', 'OPERATION', 'DATE']
    data = []
    
    index = 0
    for caisson in range(N):
        current_p = initial_p_values[caisson]
        total_days = 0
        alpha = alpha_values[caisson]
        
        while total_days < T:
            time_to_3_2 = int((current_p - 3.2) / alpha)
            if total_days + time_to_3_2 >= T:
                break
            
            operation = 'x'
            data.append([index, caisson, initial_p_values[caisson], operation, total_days + time_to_3_2])
            index += 1
            total_days += time_to_3_2 + 1
            current_p = 3.5
    
    maintenance_plan = pd.DataFrame(data, columns=columns)
    return maintenance_plan, alpha_values


def run_clustering_analysis(PARAMS, random_seed):
    """完整聚类分析函数"""
    # 生成维护计划 - 使用PARAMS中的target_daily_tasks和random_seed
    df, alpha_values_list = generate_maintenance_plan(
        N=PARAMS['P'], 
        T=PARAMS['T'], 
        random_seed=random_seed,
        target_daily_tasks=PARAMS['target_daily_tasks']
    )
    print(f"Generated {len(df)} tasks for seed {random_seed}")
    
    # alpha_values字典格式
    alpha_values = {i: alpha_values_list[i] for i in range(PARAMS['P'])}
    globals()['alpha_values'] = alpha_values
    globals()['PARAMS'] = PARAMS

    # 🔥 新增：整个聚类过程CPU时间测量
    import time
    clustering_cpu_start = time.process_time()
    
    # 聚类函数
    def clustering_CR(df_window, window_size, window_start):
        if df_window.empty or window_size < 1:
            print(f"warning: window {window_start} has no tasks")
            return df_window

        N_tasks = df_window.shape[0]
        m = gp.Model('Dynamic_Window_Clustering')
        m.setParam('OutputFlag', 0)
        
        data_pos_num = window_size + 1
        theta = m.addVars(N_tasks, data_pos_num, vtype=GRB.BINARY, name='theta')
        cluster = m.addVars(data_pos_num, vtype=GRB.BINARY, name='cluster')

        a = [0] * N_tasks
        b = [0] * N_tasks

        m.setObjective(gp.quicksum(cluster[t] for t in range(data_pos_num)), GRB.MINIMIZE)
        
        m.addConstrs((theta[n, t] <= cluster[t] for n in range(N_tasks) for t in range(data_pos_num)), 
                     name="cluster_activation")
        m.addConstrs((gp.quicksum(theta[n, t] for t in range(data_pos_num)) == 1 for n in range(N_tasks)), 
                     name="task_assignment")
        m.addConstrs((gp.quicksum(theta[n, t] for n in range(N_tasks)) <= PARAMS['Cmax'] for t in range(data_pos_num)), 
                     name="cluster_capacity")

        for n in range(N_tasks):
            csem = int(df_window.iloc[n]['CSEM'])
            local_time = df_window.iloc[n]['window_time']
            a[n] = max(0, local_time - PARAMS['advanced'])
            alpha = alpha_values[csem]
            b[n] = min(data_pos_num, (local_time + int(0.2 / alpha))//2)

        for n in range(N_tasks):
            for t in range(data_pos_num):
                m.addConstr(t * theta[n,t] + a[n] * (1 - theta[n, t]) >= a[n])
                m.addConstr(t * theta[n,t] + b[n] * (1 - theta[n, t]) <= b[n])

        m.setParam('LogToConsole', 1)
        m.setParam('OutputFlag', 1)
        m.setParam('TimeLimit', 300)
        m.setParam('MIPGap', 0.05)

        print(f"Window {window_start}: N_tasks={N_tasks}")
        m.optimize()

        if m.status != GRB.OPTIMAL:
            print(f"Failed to solve window {window_start}: {m.status}")
            return None
        
        centers = {}
        for n in range(N_tasks):
            for t in range(data_pos_num):
                if theta[n, t].x == 1.0:
                    centers[n] = t + window_start
                    break

        index_n_list = df_window.INDEX.tolist()
        for n in range(len(index_n_list)):
            df_window.loc[df_window['INDEX'] == index_n_list[n], 'CENTER'] = centers.get(n, 0)

        return df_window
    
    # 泄漏计算函数
    def calculate_gas_leakage(df, T, use_center=True):
        total_leakage = 0
        for csem in df['CSEM'].unique():
            csem_data = df[df['CSEM'] == csem]
            x_operations = csem_data[csem_data['OPERATION'] == 'x'].sort_values('DATE' if not use_center else 'CENTER')
            if x_operations.empty:
                continue
                
            alpha = alpha_values[int(csem)]

            # 任务间泄漏
            for i in range(1, len(x_operations)):
                current_time = x_operations.iloc[i][('CENTER' if use_center else 'DATE')]
                prev_time = x_operations.iloc[i-1][('CENTER' if use_center else 'DATE')]
                center_diff = current_time - prev_time - 1
                total_leakage += max(0, center_diff) * alpha * PARAMS['K']

            # 开头泄漏
            first_time = x_operations.iloc[0][('CENTER' if use_center else 'DATE')]
            total_leakage += first_time * alpha * PARAMS['K']

            # 结尾泄漏
            last_time = x_operations.iloc[-1][('CENTER' if use_center else 'DATE')]
            leak_time = max(0, T - (last_time + 1))
            total_leakage += leak_time * alpha * PARAMS['K']
        return total_leakage
    
    # 更新次要任务函数
    def update_secondary_tasks(main_df, clustered_df, window_start, window_end):
        for caisson in clustered_df["CSEM"].unique():
            tasks = main_df[
                (main_df['CSEM'] == caisson) & 
                (main_df['DATE'].between(window_start, window_end)) & 
                (main_df['OPERATION'] == 'x')
            ]
            if len(tasks) > 1:
                second_idx = tasks.index[1]
                center_time = clustered_df.loc[clustered_df['CSEM'] == caisson, 'CENTER'].iloc[0]
                alpha = alpha_values[int(caisson)]
                new_date = int(center_time + 1 + (3.5-3.2)/alpha)
                main_df.loc[second_idx, ['DATE', 'CENTER']] = [new_date, new_date]

    # 主逻辑
    df_init = df.copy()
    df_history_list = [df_init]
    sliding_windows_starts = []
    sliding_windows_ends = []
    df['CENTER'] = df['DATE']
    
    window_start = 0
    window_size = PARAMS['window_size']

    while window_start <= PARAMS['T']:
        window_end = min(window_start + window_size, PARAMS['T'])
        sliding_windows_starts.append(window_start)
        sliding_windows_ends.append(window_end)
        
        mask = (df['DATE'] >= window_start) & (df['DATE'] <= window_end) & (df['OPERATION']== 'x')
        window_df = df.loc[mask].copy()
        if window_df.empty:
            break
            
        window_df['window_time'] = window_df['DATE'] - window_start
        first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])

        clustered_df = clustering_CR(first_tasks, window_size, window_start)
        
        if clustered_df is not None:
            center_map = clustered_df.set_index('INDEX')['CENTER']
            df.loc[df['INDEX'].isin(center_map.index), 'CENTER'] = df['INDEX'].map(center_map)
            update_secondary_tasks(df, clustered_df, window_start, window_end)
        
        # 滑动窗口逻辑
        second_dates = []
        if clustered_df is not None:
            for caisson in clustered_df["CSEM"].unique():
                tasks = df[(df['CSEM'] == caisson) & 
                          (df['OPERATION'] == 'x') &
                          (df['DATE'].between(window_start, window_end))].sort_values('DATE')
                if len(tasks) > 1:
                    second_dates.append(tasks.iloc[1]['DATE'])
        
        window_start = min(second_dates) if second_dates else window_start + window_size
        df_history_list.append(df.copy())
        
        if window_end >= PARAMS['T']:
            break
    
    # 处理未分配任务
    mask_remain = (df['OPERATION'] == "x") & (df['CENTER'] == df['DATE'])
    df.loc[mask_remain, 'CENTER'] = df.loc[mask_remain, 'DATE']

    # 🔥 聚类过程CPU时间结束
    clustering_cpu_end = time.process_time()
    total_clustering_cpu_time = clustering_cpu_end - clustering_cpu_start

    # 计算泄漏
    leakage_before = calculate_gas_leakage(df_init, PARAMS['T'], use_center=False)
    leakage_after = calculate_gas_leakage(df, PARAMS['T'], use_center=True)
    reduction = leakage_before - leakage_after
    reduction_pct = (reduction / leakage_before * 100) if leakage_before > 0 else 0
    
    return {
        'final_df': df,
        'df_history_list': df_history_list,
        'sliding_windows_starts': sliding_windows_starts,
        'sliding_windows_ends': sliding_windows_ends,
        'alpha_values': alpha_values_list,
        'maintenance_plan': df_init,
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'reduction': reduction,
        'reduction_pct': reduction_pct,
        'total_clustering_cpu_time': total_clustering_cpu_time
    }


def calculate_detailed_stats(result_dict, PARAMS):
    """
    计算所有详细指标，用于CSV输出
    """
    df_orig = result_dict['maintenance_plan'].copy()
    result_df = result_dict['final_df'].copy()
    
    # 只针对x任务，确保INDEX对齐
    df_x = df_orig[df_orig['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    res_x = result_df[result_df['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    
    # CP成本计算
    time_diffs = res_x['CENTER'].values - df_x['DATE'].values
    cp_cost = np.sum(
        np.where(time_diffs < 0, np.abs(time_diffs) * PARAMS['Calpha'], time_diffs * PARAMS['Cbeta'])
    )
    
    # 聚类统计
    center_counts = res_x['CENTER'].value_counts()
    num_groups = len(center_counts)
    avg_tasks_per_group = center_counts.mean()
    
    # 时间偏移统计
    num_tasks = len(time_diffs)
    avg_shift_abs = np.mean(np.abs(time_diffs))
    max_shift_abs = np.max(np.abs(time_diffs))
    
    # 提前统计
    advanced_mask = time_diffs < 0
    avg_advance = np.mean(np.abs(time_diffs[advanced_mask])) if np.any(advanced_mask) else 0
    num_advanced = np.sum(advanced_mask)
    pct_advanced = num_advanced / num_tasks * PARAMS['Cs']
    
    # 延迟统计
    delayed_mask = time_diffs > 0
    avg_delay = np.mean(time_diffs[delayed_mask]) if np.any(delayed_mask) else 0
    num_delayed = np.sum(delayed_mask)
    pct_delayed = num_delayed / num_tasks * PARAMS['Cs']
    
    # 不变统计
    num_unchanged = np.sum(time_diffs == 0)
    pct_unchanged = num_unchanged / num_tasks * PARAMS['Cs']
    
    # 成本统计
    operation_x_count = len(df_x)
    cs_cost_before = operation_x_count * PARAMS['Cs']
    cs_cost_after = num_groups * PARAMS['Cs']
    cs_cost_reduction = cs_cost_before - cs_cost_after
    cs_cost_reduction_pct = cs_cost_reduction / cs_cost_before * 100
    total_cost = cs_cost_after + cp_cost
    
    # 泄漏统计（来自result_dict）
    leakage_before = result_dict['leakage_before']
    leakage_after = result_dict['leakage_after']
    leakage_reduction = result_dict['reduction']
    leakage_reduction_pct = result_dict['reduction_pct']

    # 🔥 整个聚类过程CPU时间
    total_clustering_cpu_time = result_dict.get('total_clustering_cpu_time', 0)
    
    # 完整统计字典（匹配你的表格列）
    stats = {
        # 基本信息
        'seed': None,  # 稍后填充
        'nb_groups': num_groups,
        'Avg_tasks/group': avg_tasks_per_group,
        
        # CP成本
        'CP_cost': cp_cost,
        
        # 时间偏移
        'Avg_shift_(abs)': avg_shift_abs,
        'Max_shift_(abs)': max_shift_abs,
        'Avg_advance_(days)': avg_advance,
        'Avg_delay_(days)': avg_delay,
        'pct_advanced_%': pct_advanced,
        'pct_delayed_%': pct_delayed,
        'pct_unchanged_%': pct_unchanged,
        'num_advanced': num_advanced,
        'num_delayed': num_delayed,
        'num_unchanged': num_unchanged,
        'total_tasks': num_tasks,
        
        # CS成本
        'CS_cost_before': cs_cost_before,
        'CS_cost_after': cs_cost_after,

        # Total成本
        'Total_cost': total_cost,
        
        # 泄漏统计
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'leakage_reduction': leakage_reduction,
        'leakage_reduction_pct': leakage_reduction_pct,

        # CPU时间
        'total_clustering_cpu_s': total_clustering_cpu_time,
    }
    
    return stats


def run_analysis_with_csv_export(PARAMS, seeds, output_filename='detailed_results.csv'):
    """
    运行多种子分析并导出完整CSV（包含所有表格列）
    """
    all_results = []
    
    for seed in seeds:
        print(f"\n🔄 Processing seed {seed}...")
        
        # 运行分析
        result_dict = run_clustering_analysis(PARAMS, random_seed=seed)
        
        # 计算详细统计
        stats = calculate_detailed_stats(result_dict, PARAMS)
        stats['seed'] = seed  # 填充seed信息
        
        all_results.append(stats)
        
        # 打印摘要
        print(f"✅ Seed {seed}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")
    
    # 转换为DataFrame并排序列顺序（匹配你的表格）
    results_df = pd.DataFrame(all_results)
    
    # 按你的截图调整列顺序
    column_order = [
        'seed', 'nb_groups', 'Avg_tasks/group',  'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost',
        'Avg_shift_(abs)', 'Max_shift_(abs)', 'Avg_advance_(days)', 'Avg_delay_(days)','pct_advanced_%', 
        'pct_delayed_%', 'pct_unchanged_%', 'leakage_before', 'leakage_after', 'leakage_reduction', 
        'leakage_reduction_pct', 'total_clustering_cpu_s'
    ]
    
    # 只保留存在的列并排序
    available_columns = [col for col in column_order if col in results_df.columns]
    results_df = results_df[available_columns]
    
    # 保存CSV
    results_df.to_csv(output_filename, index=False, float_format='%.3f')
    print(f"\n📁 完整结果已保存到: {output_filename}")
    print(f"📊 包含 {len(results_df)} 种子, {len(results_df.columns)} 个指标列")
    
    return results_df


def visualize_clustering_results(result_dict, PARAMS, seed_name="default", save_csv=True, show_plot=True):
    """可视化聚类结果"""
    result_df = result_dict['final_df']
    maintenance_plan = result_dict['maintenance_plan']
    
    # 成本计算
    operation_x_count = (result_df['OPERATION'] == 'x').sum()
    before_cost = operation_x_count * PARAMS['Cs'] 
    operation_x_center_unique_count = result_df[result_df['OPERATION'] == 'x']['CENTER'].nunique()
    after_cost = operation_x_center_unique_count * PARAMS['Cs'] 
    
    print(f"Cost before grouping: {before_cost}")
    print(f"Cost after grouping: {after_cost}")
    
    # 泄漏统计
    leakage_before = round(result_dict['leakage_before'], 3)
    leakage_after = round(result_dict['leakage_after'], 3)
    print(f"Gas leakage before: {leakage_before}")
    print(f"Gas leakage after: {leakage_after}")
    
    # 节约统计
    cost_reduction = before_cost - after_cost
    cost_reduction_pct = (cost_reduction / before_cost * 100) if before_cost > 0 else 0
    print(f"Cost reduction: {cost_reduction} ({cost_reduction_pct:.1f}%)")
    print(f"Gas leakage reduction: {result_dict['reduction']:.3f} ({result_dict['reduction_pct']:.1f}%)")
    
    # 计算统计指标（修复UnboundLocalError）
    result_df_x_count = len(result_df[result_df['OPERATION'] == 'x'])
    avg_tasks_per_cluster = result_df_x_count / operation_x_center_unique_count if operation_x_center_unique_count > 0 else 0
    
    if show_plot:
        plt.figure(figsize=(8, 7))
        
        df_original_x = maintenance_plan[maintenance_plan['OPERATION'] == 'x']
        result_df_x = result_df[result_df['OPERATION'] == 'x']
        
        df_x = df_original_x.set_index('INDEX')
        result_x = result_df_x.set_index('INDEX')
        common_indices = df_x.index.intersection(result_x.index)
        df_x = df_x.loc[common_indices].reset_index()
        result_x = result_x.loc[common_indices].reset_index()
        
        time_shift = result_x['CENTER'] - df_x['DATE']
        plt.scatter(df_x['DATE'], result_x['CENTER'], 
                   c=time_shift, cmap='RdBu_r', 
                   vmin=-max(abs(time_shift.min()), abs(time_shift.max())), 
                   vmax=max(abs(time_shift.min()), abs(time_shift.max())), 
                   alpha=0.7, s=50, edgecolors='black', linewidth=0.8)
        
        plt.colorbar(label='Time Shift (days)\nRed=delayed, Blue=advanced')
        plt.title(f"Cluster Center Distribution (Seed {seed_name})", fontsize=16, fontweight='bold')
        plt.xlabel("Original time", fontsize=14)
        plt.ylabel("Cluster center", fontsize=14)
        plt.grid(True, alpha=0.3)
        
        max_range = max(result_x['DATE'].max(), result_x['CENTER'].max() + 5)
        plt.xlim(0, max_range)
        plt.ylim(0, max_range)
        plt.plot([0, max_range], [0, max_range], color='gray', linestyle='--', linewidth=2, label='45° diagonal')
        plt.legend()
        plt.gca().set_aspect('equal')
        plt.tight_layout()
        plt.show()
        
        center_counts = result_df_x['CENTER'].value_counts().sort_index()
        print("\nCluster center usage:")
        print(center_counts)
        print(f"Total clusters used: {len(center_counts)}")
        print(f"Average tasks per cluster: {avg_tasks_per_cluster:.1f}")
    
    if save_csv:
        filename = f'optimized_plan_seed_{seed_name}.csv'
        result_df.to_csv(filename, index=False)
        print(f"\nResult saved to '{filename}'")
    
    print(f"\nSummary:")
    print(f"- Cost saved: {cost_reduction_pct:.1f}%")
    print(f"- Gas leakage reduced: {result_dict['reduction_pct']:.1f}%")
    print(f"- Clusters created: {operation_x_center_unique_count}")

    return {
        'before_cost': before_cost,
        'after_cost': after_cost,
        'cost_reduction': cost_reduction,
        'cost_reduction_pct': cost_reduction_pct,
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'clusters_used': operation_x_center_unique_count,
        'avg_tasks_per_cluster': avg_tasks_per_cluster
    }


# # 使用示例 - 生成你的表格CSV if we fixed P and iterate seeds
# PARAMS = {
#     'P': 100, 
#     'T': 180, 
#     'Cmax': 20, 
#     'window_size': 20, 
#     'advanced': 3, 
#     'target_daily_tasks': 20,  # ✅ 新增：控制任务密度参数
#     'K': 5.993, 
#     'Calpha': 5, 
#     'Cbeta': 10, 
#     'Cs': 100}

# seeds_to_test = [1]

# # name the file if sensitivity for P
# detailed_results_df = run_analysis_with_csv_export(
#     PARAMS, seeds_to_test, 
#     output_filename=f'MIMAR.csv'
# )

# # 显示前几行
# print("\n📋 CSV预览:")
# print(detailed_results_df.round(3).head())
# visualize_clustering_results(
#     run_clustering_analysis(PARAMS, random_seed=1),
#     PARAMS, seed_name=1, save_csv=False, show_plot=True)





# 不同任务密度待测试值
task_densities = [1, 10, 20, 50, 100]

# 固定模型其他参数
BASE_PARAMS = {
    'P': 100,
    'T': 180,
    'Cmax': 20,
    'window_size': 20,
    'advanced': 3,
    'K': 5.993,
    'Calpha': 5,
    'Cbeta': 10,
    'Cs': 100
}

seeds_to_test = [1]   # 可设置多个例如 range(1, 6)

all_results = []  # 用来收集所有实验结果

for td in task_densities:
    print(f"\n============================")
    print(f"🔍 Running for target_daily_tasks = {td}")
    print(f"============================\n")

    # 更新参数
    PARAMS = BASE_PARAMS.copy()
    PARAMS['target_daily_tasks'] = td

    # 定义输出文件名
    filename = f"results_p{PARAMS['P']}t{PARAMS['T']}_td{td}.csv"

    # 运行分析并导出结果
    result_df = run_analysis_with_csv_export(PARAMS, seeds_to_test, output_filename=filename)
    result_df['target_daily_tasks'] = td  # 加入密度这一列
    all_results.append(result_df)

# 汇总所有结果成一个大的DataFrame
summary_df = pd.concat(all_results, ignore_index=True)
summary_df.to_csv("summary_all_task_densities.csv", index=False)
print("\n✅ 所有密度实验已完成并保存至 summary_all_task_densities.csv")

# 预览
print(summary_df.head())




🔍 Running for target_daily_tasks = 1


🔄 Processing seed 1...
Generated 293 tasks for seed 1
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.05
Window 0: N_tasks=28
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1813 rows, 609 columns and 3495 nonzeros
Model fingerprint: 0xe25cb68b
Variable types: 0 continuous, 609 integer (609 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 14.0000000
Presolve removed 1377 rows and 190 columns
Presolve time: 0.00s
Presolved: 436 rows, 419 columns, 1444 nonzeros
Variable types: 0 continuous, 419 integer (419 binary)

Root relaxation: objective 1.150943e+00, 430 iterations, 0.00 seconds (0.00 work units)

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.05
Window 40: N_tasks=34
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2197 rows, 735 columns and 4245 nonzeros
Model fingerprint: 0xfe24a9e8
Variable types: 0 continuous, 735 integer (735 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 15.0000000
Presolve removed 1696 rows and 257 columns
Presolve time: 0.00s
Presolved: 501 rows, 478 columns, 1678 nonzeros
Variable types: 0 continuous, 478 integer (478 binary)

Root relaxation: objective 1.000000e+00, 453 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Ob

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.05
Window 140: N_tasks=39
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2517 rows, 840 columns and 4868 nonzeros
Model fingerprint: 0x5d690e11
Variable types: 0 continuous, 840 integer (840 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 14.0000000
Presolve removed 1975 rows and 329 columns
Presolve time: 0.00s
Presolved: 542 rows, 511 columns, 1861 nonzeros
Variable types: 0 continuous, 511 integer (511 binary)

Root relaxation: objective 1.263889e+00, 561 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  O

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

... (59 outputs left out)

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.05
Window 156: N_tasks=100
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 6421 rows, 2121 columns and 12400 nonzeros
Model fingerprint: 0x0466903f
Variable types: 0 continuous, 2121 integer (2121 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 12.0000000
Presolve removed 5844 rows and 1635 columns
Presolve time: 0.00s
Presolved: 577 rows, 486 columns, 1886 nonzeros
Variable types: 0 continuous, 486 integer (486 binary)
Found heuristic solution: objective 11.0000000

Root relaxation: objective 4.000000e+00, 471 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |  

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_30314/1503969297.py:201: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])


✅ Seed 1: CP_cost=42020.0, Groups=82

📁 完整结果已保存到: results_p100t180_td100.csv
📊 包含 1 种子, 19 个指标列

✅ 所有密度实验已完成并保存至 summary_all_task_densities.csv
   seed  nb_groups  Avg_tasks/group  CS_cost_before  CS_cost_after  CP_cost  \
0     1         19        15.421053           29300           1900    23660   
1     1         43        15.604651           67100           4300    14810   
2     1         82        15.768293          129300           8200    42020   
3     1         82        15.768293          129300           8200    42020   
4     1         82        15.768293          129300           8200    42020   

   Total_cost  Avg_shift_(abs)  Max_shift_(abs)  Avg_advance_(days)  \
0       25560         8.150171               19            1.760000   
1       19110         2.473920                9            1.729469   
2       50220         3.610209               17            2.080357   
3       50220         3.610209               17            2.080357   
4       50220         3.61